# GMM-Seeded Random Walker

This notebook evaluates a new fourth model without changing `main.ipynb` or any existing model. The saved healthy and tumor GMMs produce a whole-tumor posterior on central slice 80. High-confidence posterior pixels become automatic tumor and healthy seeds, and a Random Walker propagates those labels using boundaries in the four MRI modalities.

Training volumes 1-250 fit the existing GMMs, volumes 251-300 select Random-Walker parameters, and volumes 301-369 are used once for final evaluation. All new outputs are saved under model-specific names.

Required packages: `scikit-image` and `optuna`. If needed, install them in the active environment with `python -m pip install --user scikit-image optuna` before running the imports.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from config import PROJECT_ROOT
from gmm_random_walker.evaluation import evaluate_random_walker_test_set
from gmm_random_walker.model import evaluate_random_walker_volume
from gmm_random_walker.optimization import (
    PARAMETER_NAMES,
    RANDOM_WALKER_OPTIMIZATION_VERSION,
    run_random_walker_optimization,
)

%matplotlib inline

## Validation-only parameter selection

Optuna tunes the XY spatial interpolation, tumor prior scale, automatic seed thresholds and Random-Walker graph parameters. Selection prioritizes fewer validation misses, then tumor-present Dice, while requiring validation precision of at least 0.75 and at most one false positive on a tumor-free slice.

In [ ]:
PARAMETER_PATH = Path(PROJECT_ROOT) / "saved_parameters" / "gmm_random_walker_best_params.npz"
N_OPTIMIZATION_TRIALS = 40
REOPTIMIZE = False

def load_or_optimize_parameters():
    if PARAMETER_PATH.exists() and not REOPTIMIZE:
        with np.load(PARAMETER_PATH) as saved:
            missing = [name for name in PARAMETER_NAMES if name not in saved.files]
            version = int(saved["optimization_version"]) if "optimization_version" in saved.files else 0
            if not missing and version == RANDOM_WALKER_OPTIMIZATION_VERSION:
                print(f"Loaded validation-selected parameters from {PARAMETER_PATH}")
                return {name: saved[name].item() for name in PARAMETER_NAMES}
    return run_random_walker_optimization(n_trials=N_OPTIMIZATION_TRIALS)

random_walker_params = load_or_optimize_parameters()
display(pd.DataFrame(random_walker_params.items(), columns=["Parameter", "Value"]))

## Final held-out test evaluation

Run this cell only after the validation parameters are frozen. Results are written to `output/gmm_random_walker/` and do not overwrite the existing evaluation files.

In [ ]:
RW_RESULTS = evaluate_random_walker_test_set(**random_walker_params)

## Comparison with saved existing-model results

The table reads the existing result files without rerunning or modifying those models.

In [ ]:
comparison_rows = []
existing_models = {
    "Baseline GMM": "baseline_gmm_metrics.npz",
    "Spatial GMM": "spatial_gmm_metrics.npz",
    "Spatial GMM + NDI": "spatial_gmm_ndi_metrics.npz",
}
for model_name, filename in existing_models.items():
    path = Path(PROJECT_ROOT) / "output" / "evaluation_scores" / filename
    if path.exists():
        with np.load(path) as result:
            comparison_rows.append({
                "Model": model_name,
                "Dice mean": result["mean_dice"].item(),
                "Tumor-present Dice": result["tumor_present_mean_dice"].item(),
                "Precision": result["tumor_present_mean_precision"].item(),
                "Recall": result["tumor_present_mean_recall"].item(),
                "IoU mean": result["mean_iou"].item(),
                "Missed tumors": f"{result['missed_tumors_count'].item()}/{result['gt_tumors_count'].item()}",
                "Empty-slice FP": result["tumor_free_false_positives"].item(),
            })
comparison_rows.append({
    "Model": "GMM + Random Walker",
    "Dice mean": RW_RESULTS["mean_dice"],
    "Tumor-present Dice": RW_RESULTS["tumor_present_mean_dice"],
    "Precision": RW_RESULTS["tumor_present_mean_precision"],
    "Recall": RW_RESULTS["tumor_present_mean_recall"],
    "IoU mean": RW_RESULTS["mean_iou"],
    "Missed tumors": f"{RW_RESULTS['missed_tumors_count']}/{RW_RESULTS['gt_tumors_count']}",
    "Empty-slice FP": RW_RESULTS["tumor_free_false_positives"],
})
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.round(4))

## Missed tumors and qualitative inspection

In [ ]:
missed_volumes = list(map(int, RW_RESULTS["missed_volume_numbers"]))
print("Completely missed tumors:", missed_volumes)
inspection_volumes = missed_volumes[:6]
if inspection_volumes:
    fig, axes = plt.subplots(len(inspection_volumes), 5, figsize=(16, 3.2 * len(inspection_volumes)), squeeze=False)
    for row, volume in enumerate(inspection_volumes):
        details = evaluate_random_walker_volume(volume, return_details=True, **random_walker_params)
        panels = [
            (details["image"][:, :, 3], "FLAIR"),
            (details["posterior"], "GMM tumor posterior"),
            (details["tumor_seeds"], "Tumor seeds"),
            (details["prediction"], f"Prediction (Dice={details['dice']:.3f})"),
            (details["ground_truth"], "Ground truth"),
        ]
        for column, (panel, title) in enumerate(panels):
            axes[row, column].imshow(panel, cmap="gray")
            axes[row, column].set_title(f"Volume {volume}: {title}")
            axes[row, column].axis("off")
    plt.tight_layout()
    figure_path = Path(PROJECT_ROOT) / "output" / "gmm_random_walker" / "missed_tumor_inspection.png"
    plt.savefig(figure_path, dpi=220, bbox_inches="tight")
    plt.show()